# 01 — Eksplorasi Format Data (Langkah 2)

Tujuan notebook ini: **memahami format data sebelum menulis logika** kohort, label, dan fitur.
Hasilnya dipakai untuk mengisi aturan di `src/constants.py` dan dicatat di `reports/decision_log.md`.

**Aturan privasi (wajib):**
- Notebook ini sengaja **tidak menampilkan baris data pasien**. Yang ditampilkan hanya nama kolom, tipe data, jumlah, persentase, statistik ringkas, dan *pola* format (huruf diganti `A`, angka diganti `9`).
- Kategori dengan jumlah < 10 disembunyikan otomatis.
- Kalau kamu menambahkan sel sendiri, **jangan** memakai `df.head()`, `df.sample()`, atau mencetak isi `chiefcomplaint`/`report_*`.
- Sebelum commit: **Clear All Outputs** (atau jalankan `jupyter nbconvert --clear-output --inplace notebooks/01_eksplorasi.ipynb`).
- Yang boleh kamu kirim balik ke Claude: bagian **Ringkasan** di akhir notebook.

Urutan: A. Ext-ICD → B. MIMIC-IV-ED → C. Machine measurements & record list → D. Cek join → E. Ringkasan.

In [ ]:
%pip install pandas numpy

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Cari root repo (folder yang punya data/raw), supaya notebook bisa dijalankan dari notebooks/ maupun root
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
print("Root repo :", ROOT)

PATHS = {
    "ext_icd":   RAW / "records_w_diag_icd10.csv",
    "edstays":   RAW / "mimic-iv-ed" / "ed" / "edstays.csv.gz",
    "triage":    RAW / "mimic-iv-ed" / "ed" / "triage.csv.gz",
    "medrecon":  RAW / "mimic-iv-ed" / "ed" / "medrecon.csv.gz",
    "mm":        RAW / "mimic-iv-ecg" / "machine_measurements.csv",
    "mm_dict":   RAW / "mimic-iv-ecg" / "machine_measurements_data_dictionary.csv",
    "record_list": RAW / "mimic-iv-ecg" / "record_list.csv",
}
for k, p in PATHS.items():
    size = f"{p.stat().st_size/1e6:,.1f} MB" if p.exists() else "TIDAK ADA"
    print(f"{k:12s} {size:>12s}  {p.relative_to(ROOT)}")

In [ ]:
# ===== Helper: semua output hanya agregat =====
MIN_COUNT = 10          # kategori dengan jumlah < ini disembunyikan
RINGKASAN = {}          # temuan penting, dicetak di bagian E

def ringkas(df, nama):
    print(f"== {nama}: {len(df):,} baris x {df.shape[1]} kolom")
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_%": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })
    display(info)
    RINGKASAN[f"{nama}_shape"] = f"{len(df):,} x {df.shape[1]}"
    RINGKASAN[f"{nama}_kolom"] = list(df.columns)

def vc(s, top=20, normalize=False):
    # value_counts yang aman: kategori < MIN_COUNT digabung jadi '(lainnya <10)'
    c = s.value_counts(dropna=False)
    kecil = c[c < MIN_COUNT]
    c = c[c >= MIN_COUNT].head(top)
    out = c.to_frame("n")
    if len(kecil):
        out.loc[f"(disembunyikan: {len(kecil)} kategori <{MIN_COUNT})"] = kecil.sum()
    out["%"] = (out["n"] / len(s) * 100).round(2)
    return out

def pola(x, maxlen=45):
    # Ubah string jadi pola format: huruf -> A, angka -> 9, tanda baca tetap
    if pd.isna(x):
        return "<NaN>"
    x = str(x)[:maxlen]
    x = re.sub(r"[A-Za-z]", "A", x)
    x = re.sub(r"[0-9]", "9", x)
    return x

def pola_vc(s, top=8):
    return vc(s.map(pola), top=top)

def kunci_unik(df, cols, nama):
    dup = df.duplicated(subset=cols, keep=False).sum()
    print(f"[{nama}] kunci {cols}: {'UNIK' if dup == 0 else f'TIDAK UNIK ({dup:,} baris duplikat)'}")
    RINGKASAN[f"{nama}_unik_{'+'.join(cols)}"] = (dup == 0, int(dup))
    return dup == 0

def statistik_numerik(df, cols):
    q = [0, 0.001, 0.01, 0.5, 0.99, 0.999, 1]
    out = df[cols].apply(pd.to_numeric, errors="coerce").quantile(q).T
    out.columns = ["min", "p0.1", "p1", "median", "p99", "p99.9", "max"]
    out["missing_%"] = (df[cols].isna().mean() * 100).round(2)
    return out.round(2)

## A. Ext-ICD — `records_w_diag_icd10.csv`

In [ ]:
ext = pd.read_csv(PATHS["ext_icd"], low_memory=False)
ringkas(ext, "ext_icd")

### A1. Format kolom diagnosis
Yang dicari: apakah berupa string list (`['I214', ...]`), pakai titik atau tidak, pakai kutip apa, pemisahnya apa.

In [ ]:
diag_cols = [c for c in ext.columns if "diag" in c.lower()]
print("Kolom diagnosis:", diag_cols)
for c in diag_cols:
    s = ext[c]
    s_str = s.dropna().astype(str)
    print(f"\n--- {c} | dtype={s.dtype} | missing={s.isna().mean()*100:.2f}% | string kosong/[]={(s_str.str.strip().isin(['', '[]'])).sum():,}")
    p_kurung = s_str.str.startswith("[").mean() * 100
    p_titik = s_str.str.contains(r"[A-Z][0-9]{2}\.", regex=True).mean() * 100
    p_kutip1 = s_str.str.contains("'", regex=False).mean() * 100
    p_kutip2 = s_str.str.contains('"', regex=False).mean() * 100
    print(f"  diawali '[' : {p_kurung:.1f}% | ada titik setelah 3 karakter: {p_titik:.1f}% | "
          f"ada kutip tunggal: {p_kutip1:.1f}% | ada kutip ganda: {p_kutip2:.1f}%")
    display(pola_vc(s, top=5))

In [ ]:
# Parsing percobaan: ambil token kode ICD dari string, lalu lihat pola panjang kodenya
def ambil_kode(x):
    if pd.isna(x):
        return []
    return [k.replace(".", "").strip().upper() for k in re.findall(r"[A-Za-z][0-9][0-9A-Za-z\.]*", str(x))]

if "all_diag_all" in ext.columns:
    kode = ext["all_diag_all"].map(ambil_kode)
    n_kode = kode.map(len)
    print("Jumlah kode per EKG (all_diag_all):")
    display(n_kode.describe().round(2).to_frame().T)
    semua = pd.Series([k for ks in kode for k in ks])
    print("Pola kode ICD (huruf->A, angka->9):")
    display(pola_vc(semua, top=10))
    # Kode MI saja (bukan data pasien, hanya frekuensi kode)
    mi = semua[semua.str.match(r"^I2[12]")]
    print("Frekuensi kode I21/I22 di seluruh baris EKG:")
    display(vc(mi, top=30))
    RINGKASAN["kode_mi_unik"] = sorted(mi.unique().tolist())

### A2. Kolom kunci untuk filter kohort

In [ ]:
for c in ["ecg_taken_in_ed", "ecg_taken_in_hosp", "ecg_taken_in_ed_or_hosp"]:
    if c in ext.columns:
        print(f"\n{c} | dtype={ext[c].dtype}")
        display(vc(ext[c]))
        RINGKASAN[f"{c}_dtype_dan_nilai"] = (str(ext[c].dtype), ext[c].dropna().unique()[:5].tolist())

for c in ["fold", "strat_fold"]:
    if c in ext.columns:
        print(f"\n{c} | dtype={ext[c].dtype} | nilai unik: {sorted(ext[c].dropna().unique().tolist())}")
        RINGKASAN[f"{c}_nilai"] = sorted(ext[c].dropna().unique().tolist())

if "ecg_no_within_stay" in ext.columns:
    s = ext["ecg_no_within_stay"]
    print(f"\necg_no_within_stay | dtype={s.dtype} | min={s.min()} | max={s.max()} | missing={s.isna().mean()*100:.2f}%")
    display(vc(s, top=6))
    RINGKASAN["ecg_no_within_stay_min"] = s.min()

for c in ["ed_stay_id", "ed_hadm_id", "hosp_hadm_id", "subject_id", "study_id", "age"]:
    if c in ext.columns:
        s = ext[c]
        print(f"{c:15s} dtype={str(s.dtype):8s} missing={s.isna().mean()*100:6.2f}%  n_unique={s.nunique():,}")

In [ ]:
# Pasien tidak boleh muncul di lebih dari satu fold
if {"subject_id", "fold"} <= set(ext.columns):
    n_fold_per_pasien = ext.groupby("subject_id")["fold"].nunique()
    bocor = (n_fold_per_pasien > 1).sum()
    print("Pasien yang muncul di >1 fold:", bocor)
    RINGKASAN["pasien_di_banyak_fold"] = int(bocor)

# Format file_name (untuk path EKG Orang 2)
if "file_name" in ext.columns:
    print("\nPola file_name:")
    display(pola_vc(ext["file_name"], top=3))

### A3. Pratinjau alur kohort (hanya hitungan)
Ini **pratinjau**, bukan kode final. Tujuannya melihat apakah jumlah kasus cukup (gerbang ≥ 100 positif di fold 17 dan 18).

In [ ]:
def to_bool(s):
    # tangani boolean asli maupun string 'True'/'False'
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "t", "yes"])

alur = []
def catat(nama, d):
    alur.append({"langkah": nama, "n_ekg": len(d),
                 "n_kunjungan_ed": d["ed_stay_id"].nunique() if "ed_stay_id" in d else np.nan,
                 "n_pasien": d["subject_id"].nunique()})

d = ext.copy(); catat("0. semua baris", d)
d = d[to_bool(d["ecg_taken_in_ed"])]; catat("1. ecg_taken_in_ed", d)
d = d[d["ed_stay_id"].notna()]; catat("2. ed_stay_id ada", d)
d = d[d["all_diag_all"].notna() & ~d["all_diag_all"].astype(str).str.strip().isin(["", "[]"])]; catat("3. all_diag_all ada", d)
d = d[d["ecg_no_within_stay"] == d["ecg_no_within_stay"].min()]; catat("4. EKG pertama per kunjungan", d)
display(pd.DataFrame(alur))

dup_stay = d["ed_stay_id"].duplicated(keep=False).sum()
print("Baris dengan ed_stay_id ganda setelah filter 4:", dup_stay)
RINGKASAN["kohort_pratinjau"] = pd.DataFrame(alur).to_dict("records")
RINGKASAN["dup_ed_stay_setelah_filter4"] = int(dup_stay)

d["label_pratinjau"] = d["all_diag_all"].map(lambda x: any(k.startswith(("I21", "I22")) for k in ambil_kode(x))).astype(int)
per_fold = d.groupby("fold")["label_pratinjau"].agg(n="size", positif="sum")
per_fold["prevalensi_%"] = (per_fold["positif"] / per_fold["n"] * 100).round(2)
display(per_fold)
print(f"Prevalensi total: {d['label_pratinjau'].mean()*100:.2f}%")
RINGKASAN["positif_per_fold"] = per_fold["positif"].to_dict()
RINGKASAN["prevalensi_total_%"] = round(d["label_pratinjau"].mean() * 100, 2)
kohort_ids = d[["ed_stay_id", "subject_id", "study_id"]].copy()

## B. MIMIC-IV-ED v2.2
### B1. `edstays`

In [ ]:
edstays = pd.read_csv(PATHS["edstays"])
ringkas(edstays, "edstays")
kunci_unik(edstays, ["stay_id"], "edstays")
if "arrival_transport" in edstays.columns:
    print("arrival_transport:"); display(vc(edstays["arrival_transport"]))

### B2. `triage`

In [ ]:
triage = pd.read_csv(PATHS["triage"])
ringkas(triage, "triage")
kunci_unik(triage, ["stay_id"], "triage")

vital = [c for c in ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity"] if c in triage.columns]
print("\nStatistik vital (cari satuan aneh & outlier):")
display(statistik_numerik(triage, vital))

RENTANG = {"temperature": (90, 110), "heartrate": (20, 250), "resprate": (4, 60),
           "o2sat": (50, 100), "sbp": (50, 300), "dbp": (20, 200)}
luar = {}
for c, (lo, hi) in RENTANG.items():
    if c in triage.columns:
        v = pd.to_numeric(triage[c], errors="coerce")
        luar[c] = f"{((v < lo) | (v > hi)).mean()*100:.3f}%"
print("Persentase nilai di luar rentang usulan (akan jadi NaN):", luar)
RINGKASAN["triage_luar_rentang"] = luar

# Suhu: cek apakah ada yang tampak Celsius (30-45)
if "temperature" in triage.columns:
    t = pd.to_numeric(triage["temperature"], errors="coerce")
    print(f"Suhu 30–45 (kemungkinan Celsius): {((t >= 30) & (t <= 45)).sum():,} baris")

In [ ]:
# pain: berapa yang bisa jadi angka 0-10, dan pola nilai non-angka
if "pain" in triage.columns:
    p = triage["pain"]
    num = pd.to_numeric(p, errors="coerce")
    print(f"pain dtype={p.dtype} | missing={p.isna().mean()*100:.2f}% | bisa jadi angka={num.notna().mean()*100:.2f}% | angka di 0–10={num.between(0,10).mean()*100:.2f}%")
    non_num = p[p.notna() & num.isna()].astype(str).str.strip().str.lower()
    print("Nilai pain non-angka (kategori >=10 saja):")
    display(vc(non_num, top=15))
    RINGKASAN["pain_bisa_angka_%"] = round(num.notna().mean() * 100, 2)

In [ ]:
# chiefcomplaint: TIDAK dicetak. Hanya statistik dan tingkat kecocokan pola kata kunci.
CC_POLA = {
    "cc_chest_pain": r"\b(chest pain|cp|chest pressure|chest tightness|chest discomfort)\b",
    "cc_dyspnea": r"\b(sob|dyspnea|shortness of breath|difficulty breathing)\b",
    "cc_syncope": r"\b(syncope|syncopal|passed out|loc)\b",
    "cc_weakness": r"\b(weakness|fatigue|malaise)\b",
    "cc_epigastric_abd": r"\b(epigastric|abd pain|abdominal pain)\b",
    "cc_palpitations": r"\b(palpitations|palps|racing heart)\b",
    "cc_dizziness": r"\b(dizzy|dizziness|lightheaded)\b",
    "cc_nausea_vomit": r"\b(nausea|vomiting|n/v)\b",
    "cc_diaphoresis": r"\b(diaphoresis|diaphoretic|sweating)\b",
    "cc_arm_jaw_back": r"\b(arm pain|jaw pain|back pain)\b",
    "cc_explicit_mi (BUKAN FITUR)": r"\b(stemi|nstemi|mi|acs|code stemi|cath|myocardial infarction)\b",
}
if "chiefcomplaint" in triage.columns:
    cc = triage["chiefcomplaint"].astype("string").str.lower()
    print(f"chiefcomplaint missing={cc.isna().mean()*100:.2f}% | n_unique={cc.nunique():,} | panjang median={cc.str.len().median()}")
    print(f"Pemisah: koma={cc.str.contains(',', na=False).mean()*100:.1f}% | titik koma={cc.str.contains(';', na=False).mean()*100:.1f}%")
    hasil = {k: round(cc.str.contains(v, regex=True, na=False).mean() * 100, 2) for k, v in CC_POLA.items()}
    tanpa_flag = ~pd.concat([cc.str.contains(v, regex=True, na=False) for k, v in CC_POLA.items() if "explicit" not in k], axis=1).any(axis=1)
    display(pd.Series(hasil, name="% kunjungan cocok").to_frame())
    print(f"Kunjungan tanpa flag apa pun: {tanpa_flag.mean()*100:.1f}%  (kalau besar, daftar pola perlu diperluas — lihat sendiri 200 contoh acak di komputermu, jangan dibagikan)")
    RINGKASAN["cc_hit_rate_%"] = hasil
    RINGKASAN["cc_tanpa_flag_%"] = round(tanpa_flag.mean() * 100, 1)

### B3. `medrecon`

In [ ]:
medrecon = pd.read_csv(PATHS["medrecon"], low_memory=False)
ringkas(medrecon, "medrecon")
per_stay = medrecon.groupby("stay_id").size()
print(f"Kunjungan dengan medrecon: {per_stay.size:,} dari {edstays['stay_id'].nunique():,} ({per_stay.size/edstays['stay_id'].nunique()*100:.1f}%)")
print("Baris obat per kunjungan:"); display(per_stay.describe().round(1).to_frame().T)
RINGKASAN["medrecon_coverage_%"] = round(per_stay.size / edstays["stay_id"].nunique() * 100, 1)

# Kolom golongan terapi (nama & deskripsi golongan bukan data identitas)
for c in ["etcdescription", "etc_rn", "etccode"]:
    if c in medrecon.columns:
        print(f"\n{c}: n_unique={medrecon[c].nunique():,}")
if "etcdescription" in medrecon.columns:
    print("30 golongan terapi (etcdescription) paling sering:")
    display(vc(medrecon["etcdescription"], top=30))

In [ ]:
MED_POLA = {
    "med_antiplatelet": r"aspirin|clopidogrel|ticagrelor|prasugrel",
    "med_anticoagulant": r"warfarin|apixaban|rivaroxaban|dabigatran|enoxaparin",
    "med_statin": r"atorvastatin|simvastatin|rosuvastatin|pravastatin|lovastatin",
    "med_betablocker": r"metoprolol|atenolol|carvedilol|propranolol|labetalol|bisoprolol|nadolol",
    "med_acei_arb": r"lisinopril|enalapril|ramipril|captopril|benazepril|losartan|valsartan|irbesartan|olmesartan|candesartan",
    "med_nitrate": r"nitroglycerin|isosorbide",
    "med_insulin": r"insulin",
    "med_oral_antidiabetic": r"metformin|glipizide|glyburide|glimepiride|sitagliptin|pioglitazone",
    "med_loop_diuretic": r"furosemide|torsemide|bumetanide",
    "med_ccb": r"amlodipine|diltiazem|verapamil|nifedipine",
}
if "name" in medrecon.columns:
    nama = medrecon["name"].astype("string").str.lower()
    cocok = pd.concat({k: nama.str.contains(v, regex=True, na=False) for k, v in MED_POLA.items()}, axis=1)
    per_kunjungan = cocok.groupby(medrecon["stay_id"]).max().mean() * 100
    display(per_kunjungan.round(2).to_frame("% kunjungan (yang punya medrecon)"))
    RINGKASAN["med_rate_%"] = per_kunjungan.round(2).to_dict()
    # Nama obat yang tidak masuk golongan mana pun (frekuensi obat, bukan data pasien)
    tidak_cocok = nama[~cocok.any(axis=1)]
    print("40 nama obat tersering yang TIDAK masuk golongan mana pun (cek apakah ada obat jantung terlewat):")
    display(vc(tidak_cocok, top=40))

## C. MIMIC-IV-ECG: machine measurements & record list

In [ ]:
if PATHS["mm_dict"].exists():
    print("Data dictionary (metadata, aman ditampilkan):")
    display(pd.read_csv(PATHS["mm_dict"]))

mm = pd.read_csv(PATHS["mm"], low_memory=False)
ringkas(mm, "mm")
kunci_unik(mm, ["subject_id", "study_id"], "mm")
report_cols = [c for c in mm.columns if c.startswith("report_")]
print("Kolom report_*:", len(report_cols), "(DIBUANG dari fitur, disimpan terpisah untuk baseline M0)")

In [ ]:
MM_NUM = [c for c in ["rr_interval", "p_onset", "p_end", "qrs_onset", "qrs_end", "t_end", "p_axis", "qrs_axis", "t_axis"] if c in mm.columns]
print("Statistik kolom numerik MM:")
display(statistik_numerik(mm, MM_NUM))

# Cari nilai sentinel: nilai yang berulang sangat sering di ujung ekstrem
print("\nNilai paling sering per kolom (sentinel biasanya angka ekstrem yang berulang, mis. 29999 atau 0):")
sentinel = {}
for c in MM_NUM:
    top = mm[c].value_counts().head(5)
    sentinel[c] = {float(k): int(v) for k, v in top.items()}
    print(f"  {c:12s}", sentinel[c])
RINGKASAN["mm_nilai_tersering"] = sentinel

In [ ]:
# Pratinjau fitur turunan (setelah buang nilai >= 10000 yang kemungkinan sentinel)
m = mm[MM_NUM].apply(pd.to_numeric, errors="coerce")
m = m.mask(m.abs() >= 10000)
turunan = pd.DataFrame({
    "hr_ecg": 60000 / m["rr_interval"].where(m["rr_interval"] > 0),
    "pr_interval": m["qrs_onset"] - m["p_onset"],
    "qrs_duration": m["qrs_end"] - m["qrs_onset"],
    "qt_interval": m["t_end"] - m["qrs_onset"],
})
turunan["qtc_bazett"] = turunan["qt_interval"] / np.sqrt(m["rr_interval"].where(m["rr_interval"] > 0) / 1000)
print("Statistik fitur turunan (bandingkan dengan nilai klinis: QRS ~80–120 ms, QT ~350–450 ms, PR ~120–200 ms):")
display(statistik_numerik(turunan, list(turunan.columns)))
neg = {c: int((turunan[c] < 0).sum()) for c in ["pr_interval", "qrs_duration", "qt_interval"]}
print("Jumlah nilai negatif (tanda urutan onset/end bermasalah):", neg)
RINGKASAN["turunan_median"] = turunan.median().round(1).to_dict()
RINGKASAN["turunan_negatif"] = neg

In [ ]:
rl = pd.read_csv(PATHS["record_list"])
ringkas(rl, "record_list")
kunci_unik(rl, ["subject_id", "study_id"], "record_list")
for c in ["path", "file_name"]:
    if c in rl.columns:
        print(f"Pola {c}:"); display(pola_vc(rl[c], top=3))

## D. Cek join terhadap kohort pratinjau

In [ ]:
def tipe(s): return str(s.dtype)
print("Tipe kunci: ext.ed_stay_id =", tipe(ext["ed_stay_id"]), "| edstays.stay_id =", tipe(edstays["stay_id"]),
      "| triage.stay_id =", tipe(triage["stay_id"]))
print("  (ed_stay_id biasanya float karena ada NaN -> ubah ke Int64 sebelum join)\n")

ids = kohort_ids.copy()
ids["ed_stay_id"] = ids["ed_stay_id"].astype("Int64")
hasil_join = {
    "kohort -> edstays": ids["ed_stay_id"].isin(edstays["stay_id"]).mean(),
    "kohort -> triage": ids["ed_stay_id"].isin(triage["stay_id"]).mean(),
    "kohort -> medrecon": ids["ed_stay_id"].isin(medrecon["stay_id"]).mean(),
}
kunci_mm = set(zip(mm["subject_id"], mm["study_id"]))
hasil_join["kohort -> machine_measurements"] = np.mean([k in kunci_mm for k in zip(ids["subject_id"], ids["study_id"])])
kunci_rl = set(zip(rl["subject_id"], rl["study_id"]))
hasil_join["kohort -> record_list"] = np.mean([k in kunci_rl for k in zip(ids["subject_id"], ids["study_id"])])
hasil_join = {k: f"{v*100:.2f}%" for k, v in hasil_join.items()}
display(pd.Series(hasil_join, name="% kohort yang cocok").to_frame())
RINGKASAN["join_match"] = hasil_join

# Konsistensi subject_id antara Ext-ICD dan edstays untuk stay yang sama
m_ = ids.merge(edstays[["stay_id", "subject_id"]], left_on="ed_stay_id", right_on="stay_id", how="inner", suffixes=("", "_ed"))
beda = (m_["subject_id"] != m_["subject_id_ed"]).sum()
print("stay yang subject_id-nya beda antara Ext-ICD dan edstays:", beda)
RINGKASAN["subject_id_tidak_konsisten"] = int(beda)

## E. Ringkasan
Cetakan di bawah hanya berisi struktur dan angka agregat. **Ini yang boleh kamu salin dan kirim ke Claude.**
Sebelum mengirim, lihat sekali lagi dan pastikan tidak ada isi baris pasien.

In [ ]:
import json
def aman(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)
print(json.dumps(RINGKASAN, indent=1, default=aman, ensure_ascii=False))

In [ ]:
d3 = ext[ext["ecg_taken_in_ed"] & ext["ed_stay_id"].notna()]
print(vc(d3["ecg_no_within_stay"], top=8))
print(pd.crosstab(d3["ecg_no_within_stay"].clip(upper=2), d3["ecg_taken_in_hosp"]))
n_per_stay = d3.groupby("ed_stay_id").size()
print("EKG per kunjungan IGD:", n_per_stay.describe().round(2).to_dict())